In [10]:
!pip install -q -U datasets huggingface_hub faiss-cpu

In [11]:
from datasets import load_dataset
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np

In [8]:
dataset = load_dataset(
    "rajpurkar/squad",
    split="train[:500]"
)

print(dataset[0])

README.md:   0%|          | 0.00/7.62k [00:00<?, ?B/s]

plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/14.5M [00:00<?, ?B/s]

plain_text/validation-00000-of-00001.par(…):   0%|          | 0.00/1.82M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/87599 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/10570 [00:00<?, ? examples/s]

{'id': '5733be284776f41900661182', 'title': 'University_of_Notre_Dame', 'context': 'Architecturally, the school has a Catholic character. Atop the Main Building\'s gold dome is a golden statue of the Virgin Mary. Immediately in front of the Main Building and facing it, is a copper statue of Christ with arms upraised with the legend "Venite Ad Me Omnes". Next to the Main Building is the Basilica of the Sacred Heart. Immediately behind the basilica is the Grotto, a Marian place of prayer and reflection. It is a replica of the grotto at Lourdes, France where the Virgin Mary reputedly appeared to Saint Bernadette Soubirous in 1858. At the end of the main drive (and in a direct line that connects through 3 statues and the Gold Dome), is a simple, modern stone statue of Mary.', 'question': 'To whom did the Virgin Mary allegedly appear in 1858 in Lourdes France?', 'answers': {'text': ['Saint Bernadette Soubirous'], 'answer_start': [515]}}


In [9]:
# ------------------------------------
# Extract Contexts
# ------------------------------------

contexts = []

for sample in dataset:
    contexts.append(sample["context"])

print("Total Contexts:", len(contexts))

print("\nFirst Context:\n")
print(contexts[0][:500])

contexts = list(set(contexts))

print("Unique Contexts:", len(contexts))

Total Contexts: 500

First Context:

Architecturally, the school has a Catholic character. Atop the Main Building's gold dome is a golden statue of the Virgin Mary. Immediately in front of the Main Building and facing it, is a copper statue of Christ with arms upraised with the legend "Venite Ad Me Omnes". Next to the Main Building is the Basilica of the Sacred Heart. Immediately behind the basilica is the Grotto, a Marian place of prayer and reflection. It is a replica of the grotto at Lourdes, France where the Virgin Mary reputed
Unique Contexts: 74


In [8]:
# ------------------------------------
# Chunking
# ------------------------------------

chunk_size = 200

chunks = []

for context in contexts:
    words = context.split()

    for i in range(0, len(words), chunk_size):
        chunk = " ".join(words[i:i + chunk_size])
        chunks.append(chunk)

print("Total Chunks:", len(chunks))

print("\nFirst Chunk:\n")
print(chunks[0])

Total Chunks: 88

First Chunk:

Architecturally, the school has a Catholic character. Atop the Main Building's gold dome is a golden statue of the Virgin Mary. Immediately in front of the Main Building and facing it, is a copper statue of Christ with arms upraised with the legend "Venite Ad Me Omnes". Next to the Main Building is the Basilica of the Sacred Heart. Immediately behind the basilica is the Grotto, a Marian place of prayer and reflection. It is a replica of the grotto at Lourdes, France where the Virgin Mary reputedly appeared to Saint Bernadette Soubirous in 1858. At the end of the main drive (and in a direct line that connects through 3 statues and the Gold Dome), is a simple, modern stone statue of Mary.


In [11]:
# ------------------------------------
# Generate Embeddings
# ------------------------------------

print("Loading embedding model...")

model = SentenceTransformer("all-MiniLM-L6-v2")

print("Generating embeddings...")

embeddings = model.encode(
    chunks,
    show_progress_bar=True,
    convert_to_numpy=True
)

print("Embeddings Shape:", embeddings.shape)

Loading embedding model...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Generating embeddings...


Batches:   0%|          | 0/3 [00:00<?, ?it/s]

Embeddings Shape: (88, 384)


In [12]:
# ------------------------------------
# Build FAISS Vector Store
# ------------------------------------

print("Building FAISS index...")

# Get embedding dimension
dimension = embeddings.shape[1]

# Create FAISS index
index = faiss.IndexFlatL2(dimension)

# Add embeddings to index
index.add(embeddings)

print("FAISS Index Created Successfully!")

print("Total Vectors Stored:", index.ntotal)

Building FAISS index...
FAISS Index Created Successfully!
Total Vectors Stored: 88


In [15]:
# ------------------------------------
# Evaluation Input Module
# ------------------------------------

print("=" * 60)
print("        AI Response Quality Evaluator")
print("=" * 60)

question = input("Enter User Question: ")

ai_response = input("Enter AI Generated Response: ")

reference = input("Enter Reference Answer (Optional): ")

print("\n✅ Input Received Successfully!")

        AI Response Quality Evaluator
Enter User Question: tell me about the university of Notre Dame
Enter AI Generated Response: It is a university 
Enter Reference Answer (Optional): it has a catholic architecture

✅ Input Received Successfully!


In [16]:
# ------------------------------------
# Retrieve Relevant Context
# ------------------------------------

query = question

print("User Query:", query)

# Convert query into embedding
query_embedding = model.encode([query], convert_to_numpy=True)

# Search Top-3 nearest chunks
distances, indices = index.search(query_embedding, k=3)

print("\nTop 3 Retrieved Chunks:\n")

for i, idx in enumerate(indices[0], start=1):
    print(f"Result {i}:")
    print(chunks[idx])
    print("-" * 80)

User Query: tell me about the university of Notre Dame

Top 3 Retrieved Chunks:

Result 1:
Besides its prominence in sports, Notre Dame is also a large, four-year, highly residential research University, and is consistently ranked among the top twenty universities in the United States and as a major global university. The undergraduate component of the university is organized into four colleges (Arts and Letters, Science, Engineering, Business) and the Architecture School. The latter is known for teaching New Classical Architecture and for awarding the globally renowned annual Driehaus Architecture Prize. Notre Dame's graduate program has more than 50 master's, doctoral and professional degree programs offered by the five schools, with the addition of the Notre Dame Law School and a MD-PhD program offered in combination with IU medical School. It maintains a system of libraries, cultural venues, artistic and scientific museums, including Hesburgh Library and the Snite Museum of Art. Ov

In [19]:
# ------------------------------------
# Display Final Report
# ------------------------------------

print("\n" + "=" * 60)
print("               AI Response Evaluation Summary")
print("=" * 60)

print("\nUser Question:")
print(question)

print("\nAI Generated Response:")
print(ai_response)

print("\nReference Answer:")
print(reference if reference else "Not Provided")

print("\nTop 3 Retrieved Knowledge Chunks:\n")

for i, idx in enumerate(indices[0], start=1):

    print(f"Chunk {i}")
    print("-" * 50)

    print(chunks[idx])

    print()


               AI Response Evaluation Summary

User Question:
tell me about the university of Notre Dame

AI Generated Response:
It is a university 

Reference Answer:
it has a catholic architecture

Top 3 Retrieved Knowledge Chunks:

Chunk 1
--------------------------------------------------
Besides its prominence in sports, Notre Dame is also a large, four-year, highly residential research University, and is consistently ranked among the top twenty universities in the United States and as a major global university. The undergraduate component of the university is organized into four colleges (Arts and Letters, Science, Engineering, Business) and the Architecture School. The latter is known for teaching New Classical Architecture and for awarding the globally renowned annual Driehaus Architecture Prize. Notre Dame's graduate program has more than 50 master's, doctoral and professional degree programs offered by the five schools, with the addition of the Notre Dame Law School and a MD